# m35 lab · Validating a model the honest way

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

This lab builds a small classifier and then checks it the way module m35 recommends.

| Part | What you do | Data |
| --- | --- | --- |
| A | Generate labeled electrograms, local versus far-field, from the m03 model, with patient-to-patient differences | synthetic |
| B | Fit logistic regression; evaluate with patient-wise cross-validation: accuracy, AUC, calibration | synthetic |
| C | Leakage: compare record-wise and patient-wise splits for a model that memorizes | synthetic |
| D | A confidence interval for AUC that resamples patients, not records | synthetic |
| E | Flip a share of the labels and watch the measured accuracy fall to the predicted ceiling | synthetic |

**How to run.** From the course repository: `pip install -r requirements.txt`, then open this notebook in Jupyter.
In Google Colab the first code cell fetches the course code for you. The whole lab runs in under a minute.

Cells that start with `assert` are **checks**: they pass with the code as given. If you change something and a
check fails, that is information, not an error to hide. All data here are synthetic or come from the course's own
models; nothing describes a real product or patient.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Find the shared model code in labs/python/epml. In Colab, fetch the course repository first.
try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/open-ep-mapping-lab/open-ep-mapping-lab.github.io",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, KFold, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier

from epml import electrograms as eg
from epml import ml

## Part A · Labeled electrograms

A bipole records the same kind of planar front either close by (local) or far away (far-field). In the m03 model the
signal shape is set by $a = w + h$: a close front gives a sharp, large signal, a distant one a broad, small one.
Local fronts here are 0.8–4 mm away and far-field fronts 2.5–8 mm, so the ranges overlap and some records are
ambiguous. Each synthetic patient has its own recording gain and noise level, so amplitude alone is unreliable. Noise
makes small signals look sharp, so the slope is measured after a 9-point moving average. All distances, gains and
noise levels are illustrative inputs.

In [ ]:
rng = np.random.default_rng(35)
t = np.linspace(-40, 40, 401)                      # ms
dt = t[1] - t[0]
smooth = np.ones(9) / 9                            # 9-point moving average
n_pat, per = 40, 30
rows = []
for p in range(n_pat):
    gain = rng.lognormal(0.0, 0.5)                 # patient-specific gain
    noise = rng.uniform(0.005, 0.03)               # patient-specific noise
    for _ in range(per):
        local = rng.random() < 0.5
        a = rng.uniform(0.8, 4.0) if local else rng.uniform(2.5, 8.0)    # mm, overlapping ranges
        x = gain * eg.bipolar_planar(-1.0 * t, 2.0, 0.0, a) / 0.5 + noise * rng.normal(size=t.size)
        xs = np.convolve(x, smooth, mode="same")
        ptp = xs.max() - xs.min()
        sharp = np.max(np.abs(np.diff(xs))) / dt / ptp                # 1/ms, gain-free
        width = np.sum(np.abs(xs) > 0.5 * np.abs(xs).max()) * dt       # ms
        rows.append((p, int(local), np.log(ptp), sharp, width))
data = np.array(rows)
groups, y, X = data[:, 0].astype(int), data[:, 1].astype(int), data[:, 2:]
print(X.shape, "records from", n_pat, "patients; share local", y.mean().round(2))

fig, ax = plt.subplots()
ax.scatter(X[y == 1, 1], X[y == 1, 0], s=6, label="local")
ax.scatter(X[y == 0, 1], X[y == 0, 0], s=6, label="far-field")
ax.set(xlabel="sharpness (1/ms)", ylabel="log peak-to-peak", title="Two features of each electrogram")
ax.legend()
plt.show()

In [ ]:
assert X.shape == (n_pat * per, 3)
assert abs(y.mean() - 0.5) < 0.05
assert np.median(X[y == 1, 1]) > np.median(X[y == 0, 1])    # local signals are sharper

## Part B · Logistic regression, evaluated by patient

`GroupKFold` keeps every record of a patient in the same fold, so each prediction is for a patient the model never
saw. The predictions are out-of-fold probabilities; calibration uses the module's $\alpha$ and $\beta$.

In [ ]:
model = LogisticRegression(max_iter=1000)
p_hat = cross_val_predict(model, X, y, groups=groups, cv=GroupKFold(n_splits=5), method="predict_proba")[:, 1]
acc = np.mean((p_hat > 0.5) == y)
auc = roc_auc_score(y, p_hat)
alpha, beta = ml.calibration_slope_intercept(p_hat, y)
print(f"accuracy {acc:.3f}, AUC {auc:.3f} (rank formula {ml.auc_rank(p_hat, y):.3f}), "
      f"Brier {ml.brier_score(p_hat, y):.3f}, calibration slope {beta:.2f}, intercept {alpha:.2f}")

In [ ]:
assert abs(auc - ml.auc_rank(p_hat, y)) < 1e-12
assert 0.75 < auc < 0.999
assert 0.6 < beta < 1.6     # roughly calibrated out of fold

## Part C · Leakage

Now a patient-level label (for example "this patient's tissue is fibrotic", here purely synthetic) with a weak true
signal, and features that carry a strong patient signature. A one-nearest-neighbour model can recognize the patient
from the signature. With a record-wise split it often finds a sibling record and copies its label.

In [ ]:
def patient_task(n_patients=60, per_patient=10, icc=0.8, signal=1.0, dims=6, seed=3):
    r = np.random.default_rng(seed)
    yp = np.arange(n_patients) % 2
    u = r.normal(0, np.sqrt(icc), (n_patients, dims))
    Xc = np.repeat(u, per_patient, axis=0) + r.normal(0, np.sqrt(1 - icc), (n_patients * per_patient, dims))
    yc = np.repeat(yp, per_patient)
    Xc[:, 0] += signal * yc
    return Xc, yc, np.repeat(np.arange(n_patients), per_patient)

Xc, yc, gc = patient_task()
knn = KNeighborsClassifier(n_neighbors=1)
acc_record = np.mean(cross_val_predict(knn, Xc, yc, cv=KFold(5, shuffle=True, random_state=0)) == yc)
acc_patient = np.mean(cross_val_predict(knn, Xc, yc, groups=gc, cv=GroupKFold(5)) == yc)
lr = LogisticRegression(max_iter=1000)
lr_record = np.mean(cross_val_predict(lr, Xc, yc, cv=KFold(5, shuffle=True, random_state=0)) == yc)
lr_patient = np.mean(cross_val_predict(lr, Xc, yc, groups=gc, cv=GroupKFold(5)) == yc)
print(f"1-NN: record-wise {acc_record:.2f}, patient-wise {acc_patient:.2f}")
print(f"logistic regression: record-wise {lr_record:.2f}, patient-wise {lr_patient:.2f}")
print(f"effective sample size: {ml.effective_sample_size(len(yc), 10, 0.8):.0f} of {len(yc)} records")

In [ ]:
assert acc_record - acc_patient > 0.15            # memorizing model: large optimism
assert abs(lr_record - lr_patient) < acc_record - acc_patient   # simpler model: smaller gap

## Part D · A confidence interval that respects patients

Take the patient-level task from Part C and score logistic regression with a patient-wise split. Its records are
strongly clustered by patient. Resampling records treats them as independent and gives an interval that is too
narrow; resampling patients, with all of their records, gives an honest one. (For Part B's task, where labels vary
within each patient, the two intervals come out similar; try it.)

In [ ]:
def bootstrap_auc(p, yy, units, n_boot=600, seed=1):
    r = np.random.default_rng(seed)
    ids = np.unique(units)
    out = []
    for _ in range(n_boot):
        pick = r.choice(ids, size=ids.size, replace=True)
        idx = np.concatenate([np.flatnonzero(units == i) for i in pick])
        if 0 < yy[idx].mean() < 1:
            out.append(roc_auc_score(yy[idx], p[idx]))
    return np.percentile(out, [2.5, 97.5])

p_c = cross_val_predict(lr, Xc, yc, groups=gc, cv=GroupKFold(5), method="predict_proba")[:, 1]
auc_c = roc_auc_score(yc, p_c)
ci_patient = bootstrap_auc(p_c, yc, gc)
ci_record = bootstrap_auc(p_c, yc, np.arange(yc.size))
print(f"AUC {auc_c:.3f}; 95% CI resampling patients {ci_patient.round(3)}, resampling records {ci_record.round(3)}")

In [ ]:
assert ci_patient[0] <= auc_c <= ci_patient[1]
assert np.diff(ci_patient)[0] > np.diff(ci_record)[0]      # the honest interval is wider

## Part E · Label noise caps the measured score

Flip a share $\epsilon$ of the test labels at random. The module predicts a measured accuracy of
$a(1-\epsilon) + (1-a)\epsilon$ for a model whose true accuracy is $a$.

In [ ]:
rng_e = np.random.default_rng(5)
eps_values = [0.0, 0.1, 0.2, 0.3]
measured = [np.mean((p_hat > 0.5) == ml.flip_labels(y, e, rng_e)) for e in eps_values]
predicted = [ml.measured_accuracy(acc, e) for e in eps_values]
for e, m_, p_ in zip(eps_values, measured, predicted):
    print(f"eps {e:.1f}: measured {m_:.3f}, predicted {p_:.3f}")

In [ ]:
assert np.allclose(measured, predicted, atol=0.03)
assert all(m_ <= 1 - e + 0.03 for m_, e in zip(measured, eps_values))